# Hướng dẫn tải dữ liệu ERA5

Notebook này hướng dẫn cách sử dụng API của **Copernicus Climate Data Store (CDS)** để tải dữ liệu ERA5.

## Bước 1: Đăng ký và cấu hình CDS API
1. Đăng ký tài khoản tại [Copernicus Climate Data Store](https://cds.climate.copernicus.eu/user/register).
2. Đăng nhập và vào trang [API Key](https://cds.climate.copernicus.eu/how-to-api) để lấy thông tin `URL` và `Key`.
3. Tạo file `.cdsapirc` trong thư mục user của bạn (ví dụ: `C:\Users\<username>\.cdsapirc` trên Windows hoặc `~/.cdsapirc` trên Linux/macOS) với nội dung:
```text
url: https://cds.climate.copernicus.eu/api/v2
key: <UID>:<API-KEY>
```


## Bước 2: Cài đặt thư viện
Đảm bảo bạn đã cài `cdsapi`:
```bash
!pip install cdsapi
```

In [6]:
# !pip install cdsapi

In [7]:
import modal

# Modal Volume giữ dữ liệu kể cả khi laptop tắt hoặc notebook mất kết nối.
app = modal.App("era5-download")
image = modal.Image.debian_slim().pip_install("cdsapi")
volume = modal.Volume.from_name("era5-data", create_if_missing=True)


@app.function(
    image=image,
    secrets=[modal.Secret.from_name("cds-api")],
    volumes={"/data": volume},
    timeout=86400,
)
def download_era5():
    import calendar
    import os
    import zipfile
    from pathlib import Path

    import cdsapi

    client = cdsapi.Client(
        url=os.environ["CDS_URL"],
        key=os.environ["CDS_KEY"],
    )

    years = ["2023", "2024", "2025"]
    months = [f"{month:02d}" for month in range(1, 13)]
    data_dir = Path("/data/era5_data")
    data_dir.mkdir(parents=True, exist_ok=True)

    for year in years:
        for month in months:
            instant_file = data_dir / f"vietnam_era5_{year}_{month}_instant.nc"
            accum_file = data_dir / f"vietnam_era5_{year}_{month}_accum.nc"
            output_file = data_dir / f"vietnam_era5_{year}_{month}.zip"

            if instant_file.exists() and accum_file.exists():
                print(f"Đã có dữ liệu tháng {month}/{year}, bỏ qua...")
                continue

            print(f"Đang tải dữ liệu tháng {month} năm {year}...")
            days_in_month = calendar.monthrange(int(year), int(month))[1]
            days = [f"{day:02d}" for day in range(1, days_in_month + 1)]

            try:
                client.retrieve(
                    "reanalysis-era5-single-levels",
                    {
                        "product_type": "reanalysis",
                        "variable": [
                            "10m_u_component_of_wind",
                            "10m_v_component_of_wind",
                            "2m_temperature",
                            "mean_sea_level_pressure",
                            "total_precipitation",
                        ],
                        "year": year,
                        "month": month,
                        "day": days,
                        "time": [f"{hour:02d}:00" for hour in range(24)],
                        "area": [25.0, 100.0, 5.0, 120.0],
                        "format": "netcdf",
                    },
                    str(output_file),
                )

                with zipfile.ZipFile(output_file, "r") as archive:
                    for item in archive.infolist():
                        if item.filename.endswith("instant.nc"):
                            archive.extract(item, path=data_dir)
                            (data_dir / item.filename).replace(instant_file)
                        elif item.filename.endswith("accum.nc"):
                            archive.extract(item, path=data_dir)
                            (data_dir / item.filename).replace(accum_file)

                output_file.unlink(missing_ok=True)
                volume.commit()
                print(f"Hoàn tất tháng {month}/{year}.")

            except Exception as error:
                print(f"Lỗi khi tải tháng {month}/{year}: {error}")
                output_file.unlink(missing_ok=True)

print("Đã định nghĩa Modal job. Chạy cell tiếp theo để bắt đầu tải.")

Đã định nghĩa Modal job. Chạy cell tiếp theo để bắt đầu tải.


In [8]:
# Chạy job trên Modal. Có thể đóng laptop sau khi cell này đã gửi job thành công.
with app.run():
    download_era5.remote()

print("Job ERA5 đã hoàn tất trên Modal.")

Job ERA5 đã hoàn tất trên Modal.


## Bước 3: Cấu hình Modal

Cài Modal và đăng nhập một lần từ terminal:

```bash
pip install modal
modal setup
```

Tạo Secret chứa thông tin CDS API. Thay `UID:API_KEY` bằng API key thật của bạn:

```bash
modal secret create cds-api CDS_URL=https://cds.climate.copernicus.eu/api/v2 CDS_KEY=UID:API_KEY
```

Dữ liệu sẽ được lưu trong Modal Volume `era5-data`, không phụ thuộc vào máy tính cá nhân.

Kiểm tra lại dữ liệu trong Volume (Tùy chọn)

In [9]:
@app.function(volumes={"/data": volume})
def check_files():
    import os
    from pathlib import Path
    data_dir = Path("/data/era5_data")
    files = list(data_dir.glob("*.nc"))
    print(f"Tổng số file NetCDF hiện có: {len(files)}")
    for f in sorted(files):
        print(f.name)

# Chạy kiểm tra
with app.run():
    check_files.remote()

In [10]:
@app.function(volumes={"/data": volume}, timeout=3600)
def download_to_local():
    import shutil
    from pathlib import Path
    
    data_dir = Path("/data/era5_data")
    zip_path = Path("/data/era5_backup.zip")
    
    print("Đang nén dữ liệu trên Volume...")
    shutil.make_archive(str(zip_path.with_suffix("")), 'zip', data_dir)
    print("Đã nén xong! Bạn có thể dùng Modal CLI để tải file /era5_backup.zip về máy.")